# PTCG Imitation-Learning Submission

Attach a trained checkpoint from `training/train.py` and a Dataset containing `cg`. Edit only `MODEL_PATH`, `CG_PATH`, and `DECK` in the parameter cell. The model architecture is restored automatically from the checkpoint. Running all cells creates `/kaggle/working/submission.tar.gz`.

In [ ]:
from pathlib import Path
import torch

# Edit these two paths after attaching your Kaggle Datasets.
MODEL_PATH = Path('/kaggle/input/your-model-dataset/epoch-003.pt')
CG_PATH = Path('/kaggle/input/your-cg-dataset/cg')

# This is the deck used by the submitted agent, not a model-architecture setting.
DECK = [7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
       104, 104, 112, 112, 112, 112,
       646, 646, 646, 646, 647, 647, 647, 648, 648, 648,
       860, 860, 1079, 1079, 1079, 1080,
       1086, 1086, 1086, 1086, 1097, 1097, 1097, 1122, 1137,
       1152, 1152, 1152, 1152, 1182, 1182,
       1219, 1219, 1219, 1219, 1227, 1227, 1227, 1227,
       1231, 1259, 1259, 1259, 1259]
assert len(DECK) == 60
Path('deck.csv').write_text('\n'.join(map(str, DECK)) + '\n')

assert MODEL_PATH.is_file(), f'Checkpoint not found: {MODEL_PATH}'
assert CG_PATH.is_dir(), f'cg directory not found: {CG_PATH}'
assert (CG_PATH / '__init__.py').is_file(), f'Not a cg package: {CG_PATH}'
try:
    checkpoint_preview = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
except TypeError:
    checkpoint_preview = torch.load(MODEL_PATH, map_location='cpu')
assert isinstance(checkpoint_preview, dict), 'Checkpoint must be a mapping'
assert 'model' in checkpoint_preview and 'config' in checkpoint_preview, (
    'Checkpoint must contain model and config keys'
)
architecture = checkpoint_preview['config']
for key in (
    'd_model', 'd_feedforward', 'num_heads',
    'encoder_layers', 'decoder_layers', 'norm_mode',
    'summary_mlp_layers', 'card_mlp_layers',
    'option_numeric_mlp_layers',
):
    print(f'{key}: {architecture[key]}')
print(f"card_mlp_scope: {architecture.get('card_mlp_scope', 'shared')}")
for key, default in (
    ('pokemon_appear_embedding', False),
    ('bench_token_mlp_layers', 0),
    ('active_token_mlp_layers', 0),
    ('discard_token_mlp_layers', 0),
    ('hand_token_mlp_layers', 0),
    ('deck_token_mlp_layers', 0),
    ('region_token_mlp_residual', True),
):
    print(f'{key}: {architecture.get(key, default)}')
del checkpoint_preview
print('Model:', MODEL_PATH)
print('cg:', CG_PATH)
print('Deck cards:', len(DECK))

In [ ]:
%%writefile main.py
from __future__ import annotations

import os
from collections import Counter
from dataclasses import dataclass, field
from itertools import combinations
from typing import Any

import torch
from cg.api import AreaType, OptionType, all_attack, all_card_data, to_observation_class

torch.set_num_threads(1)
MAX_ACTIONS = 64
ENCODER_TOKENS = 26
POKEMON_ENCODER_TOKENS = 18
BENCH_SLOTS = 8
PLAYER_BENCH_COUNT_INDEX = 10
OWN_SUMMARY_DIM = 69
OPPONENT_SUMMARY_DIM = 71
GLOBAL_SUMMARY_DIM = 73
SELECT_TYPE_DIM = 11
SELECT_CONTEXT_DIM = 49
CARD_FEATURE_DIM = 54
CARD_TYPE_DIM = 7
ENERGY_TYPE_DIM = 12
CARD_ENERGY_TYPE_OFFSET = 7
CARD_HP_INDEX = 19
CARD_RETREAT_INDEX = 20
CARD_WEAKNESS_OFFSET = 21
CARD_RESISTANCE_OFFSET = 34
CARD_STAGE_OFFSET = 47
CARD_SPECIAL_OFFSET = 50
CARD_WEAKNESS_DIM = 13
CARD_RESISTANCE_DIM = 13
OPTION_TYPE_COUNT = 17
OPTION_CONTEXT_COUNT = 49
OPTION_NUMERIC_DIM = 76
ATTACK_FEATURE_DIM = 14
OPTION_PLAYER_OFFSET = 16
OPTION_AREA_OFFSET = 19
OPTION_IN_PLAY_AREA_OFFSET = 32
OPTION_TYPE_PLAY = 7
OPTION_TYPE_ATTACH = 8
OPTION_TYPE_EVOLVE = 9
OPTION_TYPE_RETREAT = 12
CARD_REGION_NAMES = (
    'own_bench', 'opponent_bench', 'own_active', 'opponent_active',
    'own_discard', 'opponent_discard', 'own_hand', 'opponent_hand',
    'own_deck', 'opponent_deck', 'own_prize', 'opponent_prize',
    'stadium', 'looking', 'unknown',
)
CARD_REGION_INDEX = {name: index for index, name in enumerate(CARD_REGION_NAMES)}
OWN_AREA_REGIONS = (
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['own_deck'],
    CARD_REGION_INDEX['own_hand'], CARD_REGION_INDEX['own_discard'],
    CARD_REGION_INDEX['own_active'], CARD_REGION_INDEX['own_bench'],
    CARD_REGION_INDEX['own_prize'], CARD_REGION_INDEX['stadium'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['looking'],
)
OPPONENT_AREA_REGIONS = (
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['opponent_deck'],
    CARD_REGION_INDEX['opponent_hand'], CARD_REGION_INDEX['opponent_discard'],
    CARD_REGION_INDEX['opponent_active'], CARD_REGION_INDEX['opponent_bench'],
    CARD_REGION_INDEX['opponent_prize'], CARD_REGION_INDEX['stadium'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['looking'],
)


def asset_path(name: str) -> str:
    # Kaggle executes main.py with exec(), so __file__ is not guaranteed.
    candidates = [name, os.path.join('/kaggle_simulations/agent', name)]
    module_file = globals().get('__file__')
    if module_file:
        candidates.append(os.path.join(os.path.dirname(module_file), name))
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(name)


def read_deck() -> list[int]:
    with open(asset_path('deck.csv'), encoding='utf-8') as handle:
        deck = [int(line.strip()) for line in handle if line.strip()]
    if len(deck) != 60:
        raise ValueError(f'Expected 60 cards, found {len(deck)}')
    return deck


MY_DECK = read_deck()


def build_card_feature_table(cards, card_count):
    features = torch.zeros((card_count, CARD_FEATURE_DIM), dtype=torch.float32)
    for card in cards:
        card_id = int(card.cardId)
        if not 0 <= card_id < card_count:
            continue
        card_type = int(card.cardType)
        if 0 <= card_type < CARD_TYPE_DIM:
            features[card_id, card_type] = 1
        energy_type = int(card.energyType)
        if 0 <= energy_type < ENERGY_TYPE_DIM:
            features[card_id, CARD_ENERGY_TYPE_OFFSET + energy_type] = 1
        features[card_id, CARD_HP_INDEX] = float(card.hp) / 400
        features[card_id, CARD_RETREAT_INDEX] = float(card.retreatCost) / 5
        weakness = None if card.weakness is None else int(card.weakness)
        weakness = weakness if weakness is not None and 0 <= weakness < ENERGY_TYPE_DIM else ENERGY_TYPE_DIM
        features[card_id, CARD_WEAKNESS_OFFSET + weakness] = 1
        resistance = None if card.resistance is None else int(card.resistance)
        resistance = resistance if resistance is not None and 0 <= resistance < ENERGY_TYPE_DIM else ENERGY_TYPE_DIM
        features[card_id, CARD_RESISTANCE_OFFSET + resistance] = 1
        features[card_id, CARD_STAGE_OFFSET:CARD_SPECIAL_OFFSET] = torch.tensor([
            float(card.basic), float(card.stage1), float(card.stage2),
        ])
        features[card_id, CARD_SPECIAL_OFFSET:CARD_FEATURE_DIM] = torch.tensor([
            float(card.ex), float(card.megaEx), float(card.tera), float(card.aceSpec),
        ])
    return features


def build_attack_feature_table(attacks, attack_count):
    features = torch.zeros((attack_count, ATTACK_FEATURE_DIM), dtype=torch.float32)
    for attack in attacks:
        attack_id = int(attack.attackId)
        if not 0 <= attack_id < attack_count:
            continue
        features[attack_id, 0] = float(attack.damage) / 300.0
        energies = list(attack.energies or [])
        for energy in energies:
            energy_type = int(energy)
            if 0 <= energy_type < ENERGY_TYPE_DIM:
                features[attack_id, 1 + energy_type] += 1
        features[attack_id, -1] = len(energies) / 5.0
    return features


@dataclass(frozen=True)
class ModelConfig:
    card_count: int
    attack_count: int
    encoder_size: int = 22_000
    d_model: int = 128
    num_heads: int = 2
    d_feedforward: int = 256
    encoder_layers: int = 1
    decoder_layers: int = 1
    norm_mode: str = 'postnorm'
    summary_mlp_layers: int = 1
    card_mlp_layers: int = 1
    option_numeric_mlp_layers: int = 1
    card_mlp_scope: str = 'shared'
    pokemon_appear_embedding: bool = False
    bench_token_mlp_layers: int = 0
    active_token_mlp_layers: int = 0
    discard_token_mlp_layers: int = 0
    hand_token_mlp_layers: int = 0
    deck_token_mlp_layers: int = 0
    region_token_mlp_residual: bool = True

    def __post_init__(self):
        if self.card_mlp_scope not in {'shared', 'region'}:
            raise ValueError('card_mlp_scope must be shared or region')
        for name in (
            'bench_token_mlp_layers', 'active_token_mlp_layers',
            'discard_token_mlp_layers', 'hand_token_mlp_layers',
            'deck_token_mlp_layers',
        ):
            if getattr(self, name) < 0:
                raise ValueError(f'{name} must be >= 0')


def projection_mlp(input_dim, d_model, layers):
    if layers < 1:
        raise ValueError('projection MLP must contain at least one layer')
    modules = [torch.nn.Linear(input_dim, d_model)]
    for _ in range(layers - 1):
        modules.extend([torch.nn.ReLU(), torch.nn.Linear(d_model, d_model)])
    return modules[0] if len(modules) == 1 else torch.nn.Sequential(*modules)


def fill_card_range(card_mapping, region_mapping, start, card_count, region):
    end = start + card_count
    card_mapping[start:end] = torch.arange(card_count)
    region_mapping[start:end] = region
    return end


def encoder_card_mappings(config):
    card_mapping = torch.full((config.encoder_size,), config.card_count, dtype=torch.long)
    region_mapping = torch.full(
        (config.encoder_size,), CARD_REGION_INDEX['unknown'], dtype=torch.long
    )
    position = 0
    field_regions = (
        CARD_REGION_INDEX['own_bench'], CARD_REGION_INDEX['opponent_bench'],
        CARD_REGION_INDEX['own_active'], CARD_REGION_INDEX['opponent_active'],
    )
    for region in field_regions:
        position += 2
        for _ in range(3):
            position = fill_card_range(
                card_mapping, region_mapping, position, config.card_count, region
            )
    zone_regions = (
        CARD_REGION_INDEX['own_discard'], CARD_REGION_INDEX['opponent_discard'],
        CARD_REGION_INDEX['own_hand'], CARD_REGION_INDEX['own_deck'],
        CARD_REGION_INDEX['stadium'],
    )
    for region in zone_regions:
        position = fill_card_range(
            card_mapping, region_mapping, position, config.card_count, region
        )
    return card_mapping, region_mapping


class CardAwareEmbeddingBag(torch.nn.EmbeddingBag):
    def __init__(
        self, num_embeddings, embedding_dim, index_to_card_id, index_to_card_region
    ):
        super().__init__(num_embeddings, embedding_dim, mode='sum')
        self.register_buffer('index_to_card_id', index_to_card_id, persistent=False)
        self.register_buffer(
            'index_to_card_region', index_to_card_region, persistent=False
        )

    def forward(self, indices, offsets, weights, projected_card_features):
        learned = super().forward(indices, offsets, per_sample_weights=weights)
        card_ids = self.index_to_card_id[indices]
        if projected_card_features.ndim == 2:
            static_indices = card_ids
            static_table = projected_card_features
        else:
            region_ids = self.index_to_card_region[indices]
            card_vocabulary = projected_card_features.size(1)
            static_indices = region_ids * card_vocabulary + card_ids
            static_table = projected_card_features.flatten(0, 1)
        static_weights = None if weights is None else weights.to(projected_card_features.dtype)
        static = torch.nn.functional.embedding_bag(
            static_indices, static_table, offsets,
            mode='sum', per_sample_weights=static_weights,
        )
        return learned + static


class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_feedforward, norm_mode):
        super().__init__()
        self.prenorm = norm_mode == 'prenorm'
        self.attention = torch.nn.MultiheadAttention(d_model, num_heads)
        self.fc1 = torch.nn.Linear(d_model, d_feedforward)
        self.fc2 = torch.nn.Linear(d_feedforward, d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)

    def forward(self, x, encoder_out, encoder_padding_mask):
        if self.prenorm:
            query = self.norm1(x)
            attended, _ = self.attention(
                query, encoder_out, encoder_out,
                key_padding_mask=encoder_padding_mask, need_weights=False
            )
            x = x + attended
            return x + self.fc2(torch.nn.functional.relu(self.fc1(self.norm2(x))))
        y, _ = self.attention(
            x, encoder_out, encoder_out,
            key_padding_mask=encoder_padding_mask, need_weights=False
        )
        residual = self.norm1(x + y)
        y = self.fc2(torch.nn.functional.relu(self.fc1(residual)))
        return self.norm2(residual + y)


class PTCGTransformer(torch.nn.Module):
    def __init__(self, config, card_feature_table, attack_feature_table):
        super().__init__()
        self.config = config
        self.encoder_token_count = ENCODER_TOKENS
        self.register_buffer('card_feature_table', card_feature_table)
        self.register_buffer('attack_feature_table', attack_feature_table)
        self.card_feature_projection = None
        self.card_feature_projections = None
        if config.card_mlp_layers > 0 and config.card_mlp_scope == 'shared':
            self.card_feature_projection = projection_mlp(
                CARD_FEATURE_DIM, config.d_model, config.card_mlp_layers
            )
        elif config.card_mlp_layers > 0:
            self.card_feature_projections = torch.nn.ModuleDict({
                name: projection_mlp(
                    CARD_FEATURE_DIM, config.d_model, config.card_mlp_layers
                )
                for name in CARD_REGION_NAMES
            })
        self.own_summary_projection = projection_mlp(
            OWN_SUMMARY_DIM, config.d_model, config.summary_mlp_layers
        )
        self.opponent_summary_projection = projection_mlp(
            OPPONENT_SUMMARY_DIM, config.d_model, config.summary_mlp_layers
        )
        self.global_summary_projection = projection_mlp(
            GLOBAL_SUMMARY_DIM, config.d_model, config.summary_mlp_layers
        )
        encoder_card_ids, encoder_card_regions = encoder_card_mappings(config)
        prenorm = config.norm_mode == 'prenorm'
        self.encoder_bag = CardAwareEmbeddingBag(
            config.encoder_size, config.d_model,
            encoder_card_ids, encoder_card_regions,
        )
        self.pokemon_appear_embedding = (
            torch.nn.Embedding(3, config.d_model, padding_idx=0)
            if config.pokemon_appear_embedding else None
        )
        self.own_bench_token_mlp = self.make_token_mlp(config.bench_token_mlp_layers)
        self.opponent_bench_token_mlp = self.make_token_mlp(config.bench_token_mlp_layers)
        self.own_active_token_mlp = self.make_token_mlp(config.active_token_mlp_layers)
        self.opponent_active_token_mlp = self.make_token_mlp(config.active_token_mlp_layers)
        self.own_discard_token_mlp = self.make_token_mlp(config.discard_token_mlp_layers)
        self.opponent_discard_token_mlp = self.make_token_mlp(config.discard_token_mlp_layers)
        self.own_hand_token_mlp = self.make_token_mlp(config.hand_token_mlp_layers)
        self.own_deck_token_mlp = self.make_token_mlp(config.deck_token_mlp_layers)
        self.register_buffer(
            'own_area_card_regions', torch.tensor(OWN_AREA_REGIONS), persistent=False
        )
        self.register_buffer(
            'opponent_area_card_regions',
            torch.tensor(OPPONENT_AREA_REGIONS), persistent=False,
        )
        layer = torch.nn.TransformerEncoderLayer(
            config.d_model, config.num_heads, config.d_feedforward,
            dropout=0, norm_first=prenorm
        )
        self.encoder = torch.nn.TransformerEncoder(
            layer, config.encoder_layers,
            norm=torch.nn.LayerNorm(config.d_model) if prenorm else None,
            enable_nested_tensor=False,
        )
        self.option_type_embedding = torch.nn.Embedding(OPTION_TYPE_COUNT, config.d_model)
        self.option_context_embedding = torch.nn.Embedding(OPTION_CONTEXT_COUNT, config.d_model)
        self.option_candidate_embedding = torch.nn.Embedding(
            config.card_count + 1, config.d_model, padding_idx=config.card_count
        )
        self.option_target_embedding = torch.nn.Embedding(
            config.card_count + 1, config.d_model, padding_idx=config.card_count
        )
        self.option_attack_embedding = torch.nn.Embedding(
            config.attack_count + 1, config.d_model, padding_idx=config.attack_count
        )
        self.option_numeric_projection = projection_mlp(
            OPTION_NUMERIC_DIM, config.d_model, config.option_numeric_mlp_layers
        )
        self.attack_feature_projection = torch.nn.Linear(ATTACK_FEATURE_DIM, config.d_model)
        self.no_action_embedding = torch.nn.Parameter(torch.zeros(config.d_model))
        self.decoder = torch.nn.ModuleList(
            DecoderLayer(
                config.d_model, config.num_heads,
                config.d_feedforward, config.norm_mode,
            )
            for _ in range(config.decoder_layers)
        )
        self.decoder_fc = torch.nn.Linear(config.d_model, 1)

    def make_token_mlp(self, layers):
        if layers == 0:
            return None
        return projection_mlp(self.config.d_model, self.config.d_model, layers)

    def apply_token_mlp(self, tokens, mlp):
        if mlp is None:
            return tokens
        transformed = mlp(tokens)
        if self.config.region_token_mlp_residual:
            return tokens + transformed
        return transformed

    def apply_region_token_mlps(self, encoded):
        return torch.cat((
            self.apply_token_mlp(encoded[:, 0:8], self.own_bench_token_mlp),
            self.apply_token_mlp(encoded[:, 8:16], self.opponent_bench_token_mlp),
            self.apply_token_mlp(encoded[:, 16:17], self.own_active_token_mlp),
            self.apply_token_mlp(encoded[:, 17:18], self.opponent_active_token_mlp),
            encoded[:, 18:20],
            self.apply_token_mlp(encoded[:, 20:21], self.own_discard_token_mlp),
            self.apply_token_mlp(encoded[:, 21:22], self.opponent_discard_token_mlp),
            self.apply_token_mlp(encoded[:, 22:23], self.own_hand_token_mlp),
            self.apply_token_mlp(encoded[:, 23:24], self.own_deck_token_mlp),
            encoded[:, 24:26],
        ), dim=1)

    def project_card_features(self):
        if (
            self.card_feature_projection is None
            and self.card_feature_projections is None
        ):
            projected = self.card_feature_table.new_zeros((
                self.config.card_count, self.config.d_model
            ))
        elif self.card_feature_projection is not None:
            projected = self.card_feature_projection(self.card_feature_table)
        else:
            projected = torch.stack([
                self.card_feature_projections[name](self.card_feature_table)
                for name in CARD_REGION_NAMES
            ])
        if projected.ndim == 2:
            return torch.cat([
                projected, projected.new_zeros((1, self.config.d_model))
            ])
        return torch.cat([
            projected,
            projected.new_zeros((projected.size(0), 1, self.config.d_model)),
        ], dim=1)

    def decoder_card_regions(self, categorical, numeric):
        option_types = categorical[:, 0]
        player_relation = numeric[
            :, OPTION_PLAYER_OFFSET:OPTION_PLAYER_OFFSET + 3
        ].argmax(dim=1)
        areas = numeric[
            :, OPTION_AREA_OFFSET:OPTION_AREA_OFFSET + 13
        ].argmax(dim=1)
        in_play_areas = numeric[
            :, OPTION_IN_PLAY_AREA_OFFSET:OPTION_IN_PLAY_AREA_OFFSET + 13
        ].argmax(dim=1)
        candidate_regions = self.own_area_card_regions[areas]
        candidate_regions = torch.where(
            player_relation == 2,
            self.opponent_area_card_regions[areas], candidate_regions,
        )
        candidate_regions = torch.where(
            option_types == OPTION_TYPE_PLAY,
            torch.full_like(candidate_regions, CARD_REGION_INDEX['own_hand']),
            candidate_regions,
        )
        target_regions = torch.full_like(
            candidate_regions, CARD_REGION_INDEX['unknown']
        )
        has_in_play_target = (
            (option_types == OPTION_TYPE_ATTACH)
            | (option_types == OPTION_TYPE_EVOLVE)
        )
        target_regions = torch.where(
            has_in_play_target, self.own_area_card_regions[in_play_areas],
            target_regions,
        )
        target_regions = torch.where(
            option_types == OPTION_TYPE_RETREAT,
            torch.full_like(target_regions, CARD_REGION_INDEX['own_active']),
            target_regions,
        )
        return candidate_regions, target_regions

    def project_attack_features(self):
        projected = self.attack_feature_projection(self.attack_feature_table)
        return torch.cat([
            projected, projected.new_zeros((1, self.config.d_model))
        ])

    def encode_options(self, categorical, numeric, card_static, attack_static):
        candidate_ids = categorical[:, 2]
        target_ids = categorical[:, 3]
        attack_ids = categorical[:, 4]
        if card_static.ndim == 3:
            candidate_regions, target_regions = self.decoder_card_regions(
                categorical, numeric
            )
            candidate_static = card_static[candidate_regions, candidate_ids]
            target_static = card_static[target_regions, target_ids]
        else:
            candidate_static = card_static[candidate_ids]
            target_static = card_static[target_ids]
        return (
            self.option_type_embedding(categorical[:, 0])
            + self.option_context_embedding(categorical[:, 1])
            + self.option_candidate_embedding(candidate_ids)
            + self.option_target_embedding(target_ids)
            + self.option_attack_embedding(attack_ids)
            + self.option_numeric_projection(numeric)
            + candidate_static
            + target_static
            + attack_static[attack_ids]
        )

    def combine_actions(self, option_embeddings, action_index, action_offset):
        if action_index.numel() == 0:
            combined = option_embeddings.new_zeros((
                action_offset.numel() - 1, self.config.d_model
            ))
        else:
            combined = torch.nn.functional.embedding_bag(
                action_index, option_embeddings, action_offset,
                mode='sum', include_last_offset=True,
            )
        empty = action_offset[1:] == action_offset[:-1]
        return combined + empty.unsqueeze(1) * self.no_action_embedding

    @staticmethod
    def _encoder_padding_mask(own_summary, opponent_summary):
        slots = torch.arange(BENCH_SLOTS, device=own_summary.device)
        own_count = torch.round(
            own_summary[:, PLAYER_BENCH_COUNT_INDEX] * BENCH_SLOTS
        ).to(torch.long).clamp(0, BENCH_SLOTS)
        opponent_count = torch.round(
            opponent_summary[:, PLAYER_BENCH_COUNT_INDEX] * BENCH_SLOTS
        ).to(torch.long).clamp(0, BENCH_SLOTS)
        own_padding = slots.unsqueeze(0) >= own_count.unsqueeze(1)
        opponent_padding = slots.unsqueeze(0) >= opponent_count.unsqueeze(1)
        fixed_tokens = torch.zeros(
            (own_summary.size(0), ENCODER_TOKENS - 2 * BENCH_SLOTS),
            dtype=torch.bool, device=own_summary.device,
        )
        return torch.cat((own_padding, opponent_padding, fixed_tokens), dim=1)

    def forward(self, index_encoder, value_encoder, offset_encoder,
                pokemon_appear, own_summary, opponent_summary, global_summary,
                option_categorical, option_numeric,
                action_option_index, action_option_offset):
        cfg = self.config
        projected_card_features = self.project_card_features()
        projected_attack_features = self.project_attack_features()
        encoded = self.encoder_bag(
            index_encoder, offset_encoder, value_encoder, projected_card_features
        )
        batch_size = own_summary.size(0)
        encoded = encoded.reshape(batch_size, ENCODER_TOKENS, cfg.d_model)
        if self.pokemon_appear_embedding is not None:
            pokemon_tokens = (
                encoded[:, :POKEMON_ENCODER_TOKENS]
                + self.pokemon_appear_embedding(pokemon_appear)
            )
            encoded = torch.cat((
                pokemon_tokens, encoded[:, POKEMON_ENCODER_TOKENS:]
            ), dim=1)
        encoded = torch.cat((
            encoded[:, :18], self.own_summary_projection(own_summary).unsqueeze(1),
            self.opponent_summary_projection(opponent_summary).unsqueeze(1),
            encoded[:, 20:25], self.global_summary_projection(global_summary).unsqueeze(1),
        ), dim=1)
        encoded = self.apply_region_token_mlps(encoded).transpose(0, 1)
        encoder_padding_mask = self._encoder_padding_mask(
            own_summary, opponent_summary
        )
        encoder_out = self.encoder(
            encoded, src_key_padding_mask=encoder_padding_mask
        )
        option_embeddings = self.encode_options(
            option_categorical, option_numeric,
            projected_card_features, projected_attack_features,
        )
        policy = self.combine_actions(
            option_embeddings, action_option_index, action_option_offset
        )
        policy = policy.reshape(batch_size, -1, cfg.d_model).transpose(0, 1)
        for layer in self.decoder:
            policy = layer(policy, encoder_out, encoder_padding_mask)
        return self.decoder_fc(policy).transpose(0, 1).reshape(batch_size, -1)


@dataclass
class SparseVector:
    index: list[int] = field(default_factory=list)
    value: list[float] = field(default_factory=list)
    offset: list[int] = field(default_factory=list)
    pos: int = 0

    def add(self, index, value):
        if float(value) != 0.0:
            self.index.append(self.pos + int(index))
            self.value.append(float(value))

    def add_pos(self, count):
        self.pos += count

    def add_single(self, value):
        self.add(0, value)
        self.pos += 1

    def word_start(self):
        self.offset.append(len(self.index))


def enumerate_actions(option_count, min_count, max_count):
    actions = []
    for count in range(max_count, min_count - 1, -1):
        for selection in combinations(range(option_count), count):
            actions.append(list(selection))
            if len(actions) == MAX_ACTIONS:
                return actions
    return actions


def add_card(sv, card, card_count):
    if card is not None:
        sv.add(card.id, 1)
    sv.add_pos(card_count)


def add_cards(sv, cards, weight, card_count):
    if cards is not None:
        for card in cards:
            sv.add(card.id, weight)
    sv.add_pos(card_count)


def add_pokemon(sv, pokemon, card_count):
    if pokemon is None:
        sv.add_single(1)
        sv.add_pos(1 + 3 * card_count)
        return
    sv.add_single(0)
    sv.add_single(pokemon.hp / 400)
    add_card(sv, pokemon, card_count)
    add_cards(sv, pokemon.tools, 1, card_count)
    add_cards(sv, pokemon.energyCards, 0.5, card_count)


def build_numeric_catalog(cards, attacks, card_count):
    card_features = build_card_feature_table(cards, card_count)
    attack_count = max((int(attack.attackId) for attack in attacks), default=-1) + 1
    attack_damage = torch.zeros(attack_count, dtype=torch.float32)
    for attack in attacks:
        attack_damage[int(attack.attackId)] = float(attack.damage) / 300.0
    card_attacks = [()] * card_count
    for card in cards:
        if 0 <= int(card.cardId) < card_count:
            card_attacks[int(card.cardId)] = tuple(int(value) for value in card.attacks)
    return card_features, attack_damage, tuple(card_attacks)


def one_hot(index, size, name):
    index = int(index)
    if not 0 <= index < size:
        raise ValueError(f'{name}={index} is outside [0, {size})')
    result = [0.0] * size
    result[index] = 1.0
    return result


def active_card(player):
    return player.active[0] if player.active else None


def player_summary(player, card_features, attack_damage, card_attacks):
    active = active_card(player)
    bench = list(player.bench[:8])
    bench_energy = sum(len(pokemon.energyCards) for pokemon in bench)
    bench_hp = sum(float(pokemon.hp) for pokemon in bench)
    bench_max_hp = sum(float(pokemon.maxHp) for pokemon in bench)
    retreat_cost = max_attack_damage = attack_count = 0.0
    weakness_norm = resistance_norm = 0.0
    if active is not None and 0 <= int(active.id) < len(card_features):
        row = card_features[int(active.id)]
        retreat_cost = float(row[CARD_RETREAT_INDEX])
        weakness_norm = float(torch.argmax(row[CARD_WEAKNESS_OFFSET:CARD_WEAKNESS_OFFSET + CARD_WEAKNESS_DIM])) / 12.0
        resistance_norm = float(torch.argmax(row[CARD_RESISTANCE_OFFSET:CARD_RESISTANCE_OFFSET + CARD_RESISTANCE_DIM])) / 12.0
        attacks = card_attacks[int(active.id)]
        attack_count = len(attacks) / 4.0
        max_attack_damage = max((float(attack_damage[a]) for a in attacks if 0 <= a < len(attack_damage)), default=0.0)
    result = [player.deckCount / 60.0, player.handCount / 20.0, len(player.discard) / 60.0]
    result.extend(one_hot(len(player.prize), 7, 'prize_count'))
    result.extend([
        len(bench) / 8.0, float(active is not None),
        float(bool(player.poisoned)), float(bool(player.burned)),
        float(bool(player.asleep)), float(bool(player.paralyzed)), float(bool(player.confused)),
        float(active.hp) / 400.0 if active is not None else 0.0,
        float(active.maxHp) / 400.0 if active is not None else 0.0,
        len(active.energyCards) / 10.0 if active is not None else 0.0,
        len(active.tools) / 4.0 if active is not None else 0.0,
        len(active.preEvolution) / 2.0 if active is not None else 0.0,
        retreat_cost, max_attack_damage, attack_count, weakness_norm, resistance_norm,
    ])
    for slot in range(8):
        if slot < len(bench):
            pokemon = bench[slot]
            result.extend([1.0, float(pokemon.hp) / 400.0, len(pokemon.energyCards) / 5.0])
        else:
            result.extend([0.0, 0.0, 0.0])
    result.extend([bench_energy / 32.0, bench_hp / 3200.0, bench_max_hp / 3200.0])
    return result


def visible_own_cards(player):
    visible = Counter()
    def add(card):
        if card is not None:
            visible[int(card.id)] += 1
    def add_pokemon(pokemon):
        if pokemon is None:
            return
        add(pokemon)
        for attached in (pokemon.energyCards, pokemon.tools, pokemon.preEvolution):
            for card in attached:
                add(card)
    for card in player.hand or []:
        add(card)
    for card in player.discard:
        add(card)
    add_pokemon(active_card(player))
    for pokemon in player.bench:
        add_pokemon(pokemon)
    return visible


def deck_remaining_summary(deck, player, card_features):
    remaining = Counter(map(int, deck))
    remaining.subtract(visible_own_cards(player))
    remaining = Counter({card_id: max(0, count) for card_id, count in remaining.items()})
    total = float(sum(remaining.values()))
    type_counts = [0.0] * CARD_TYPE_DIM
    stage_counts = [0.0, 0.0, 0.0]
    has_ex = has_mega = 0.0
    for card_id, count in remaining.items():
        if count <= 0 or not 0 <= card_id < len(card_features):
            continue
        row = card_features[card_id]
        card_type = int(torch.argmax(row[:CARD_TYPE_DIM]))
        type_counts[card_type] += count
        for stage in range(3):
            if row[CARD_STAGE_OFFSET + stage] > 0.5:
                stage_counts[stage] += count
        has_ex = max(has_ex, float(row[CARD_SPECIAL_OFFSET] > 0.5))
        has_mega = max(has_mega, float(row[CARD_SPECIAL_OFFSET + 1] > 0.5))
    pokemon, energy = type_counts[0], type_counts[5] + type_counts[6]
    return ([value / 4.0 for value in type_counts]
            + [value / 4.0 for value in stage_counts] + [has_ex, has_mega]
            + [total / 60.0, pokemon / max(total, 1.0), energy / max(total, 1.0)])


def opponent_revealed_summary(player, card_features):
    revealed = list(player.discard)
    def add_pokemon(pokemon):
        if pokemon is not None:
            revealed.append(pokemon)
            revealed.extend(pokemon.energyCards)
            revealed.extend(pokemon.tools)
            revealed.extend(pokemon.preEvolution)
    add_pokemon(active_card(player))
    for pokemon in player.bench:
        add_pokemon(pokemon)
    card_ids = [int(card.id) for card in revealed if 0 <= int(card.id) < len(card_features)]
    type_counts = [0.0] * CARD_TYPE_DIM
    has_ex = has_mega = has_tera = 0.0
    pokemon_count = 0
    average_hp = average_stage = 0.0
    for card_id in card_ids:
        row = card_features[card_id]
        card_type = int(torch.argmax(row[:CARD_TYPE_DIM]))
        type_counts[card_type] += 1.0
        if card_type == 0:
            pokemon_count += 1
            average_hp += float(row[CARD_HP_INDEX])
            has_ex = max(has_ex, float(row[CARD_SPECIAL_OFFSET] > 0.5))
            has_mega = max(has_mega, float(row[CARD_SPECIAL_OFFSET + 1] > 0.5))
            has_tera = max(has_tera, float(row[CARD_SPECIAL_OFFSET + 2] > 0.5))
            average_stage += float(row[CARD_STAGE_OFFSET + 1]) + 2.0 * float(row[CARD_STAGE_OFFSET + 2])
    if pokemon_count:
        average_hp /= pokemon_count
        average_stage /= pokemon_count
    energy_in_play = sum(len(p.energyCards) for p in ([active_card(player)] + list(player.bench)) if p is not None)
    return ([value / 10.0 for value in type_counts] + [has_ex, has_mega, has_tera]
            + [pokemon_count / 5.0, len(card_ids) / 20.0, average_hp,
               average_stage / 2.0, len(player.discard) / 20.0,
               len(player.bench) / 8.0, energy_in_play / 10.0])


def global_summary(obs, yours):
    state, select = obs.current, obs.select
    first_relative = -1.0 if int(state.firstPlayer) < 0 else float(int(state.firstPlayer) == yours)
    result = [state.turn / 100.0, state.turnActionCount / 100.0, first_relative,
              float(bool(state.supporterPlayed)), float(bool(state.stadiumPlayed)),
              float(bool(state.energyAttached)), float(bool(state.retreated)), float(yours)]
    result.extend(one_hot(int(select.type), SELECT_TYPE_DIM, 'select.type'))
    result.extend(one_hot(int(select.context), SELECT_CONTEXT_DIM, 'select.context'))
    result.extend([select.minCount / 6.0, select.maxCount / 6.0,
                   select.remainDamageCounter / 20.0, select.remainEnergyCost / 10.0,
                   len(select.option) / 64.0])
    return result


def encoder_features(obs, deck, card_count, catalog):
    card_features, attack_damage, card_attacks = catalog
    state, yours, sv = obs.current, obs.current.yourIndex, SparseVector()
    players = [state.players[yours], state.players[1 - yours]]
    pokemon_appear = []
    for player in players:
        for slot in range(8):
            pokemon = player.bench[slot] if slot < len(player.bench) else None
            pokemon_appear.append(
                0 if pokemon is None else 2 if bool(pokemon.appearThisTurn) else 1
            )
            sv.word_start()
            pos = sv.pos
            add_pokemon(sv, pokemon, card_count)
            if slot != 7:
                sv.pos = pos
    for player in players:
        pokemon = active_card(player)
        pokemon_appear.append(
            0 if pokemon is None else 2 if bool(pokemon.appearThisTurn) else 1
        )
        sv.word_start()
        add_pokemon(sv, pokemon, card_count)
    sv.word_start()
    sv.word_start()
    sv.word_start()
    add_cards(sv, players[0].discard, 0.25, card_count)
    sv.word_start()
    add_cards(sv, players[1].discard, 0.25, card_count)
    sv.word_start()
    add_cards(sv, players[0].hand, 0.25, card_count)
    sv.word_start()
    for card_id in deck:
        sv.add(card_id, 0.25)
    sv.add_pos(card_count)
    sv.word_start()
    add_cards(sv, state.stadium, 1.0, card_count)
    sv.word_start()
    own = player_summary(players[0], card_features, attack_damage, card_attacks)
    own.extend(deck_remaining_summary(deck, players[0], card_features))
    opponent = player_summary(players[1], card_features, attack_damage, card_attacks)
    opponent.extend(opponent_revealed_summary(players[1], card_features))
    return sv, pokemon_appear, own, opponent, global_summary(obs, yours)


def optional_int(value, default=0):
    return default if value is None else int(value)


def set_one_hot(values, offset, size, index, name):
    if not 0 <= index < size:
        raise ValueError(f'{name} index {index} is outside [0, {size})')
    values[offset + index] = 1.0


def area_card(obs, area, index, player_index):
    player = obs.current.players[player_index]
    mapping = {
        AreaType.DECK: obs.select.deck, AreaType.HAND: player.hand,
        AreaType.DISCARD: player.discard, AreaType.ACTIVE: player.active,
        AreaType.BENCH: player.bench, AreaType.PRIZE: player.prize,
        AreaType.STADIUM: obs.current.stadium, AreaType.LOOKING: obs.current.looking,
    }
    cards = list(mapping.get(area) or [])
    position = optional_int(index, -1)
    return cards[position] if 0 <= position < len(cards) else None


def valid_card_id(card, card_count):
    if card is None:
        return card_count
    card_id = int(card.id)
    return card_id if 0 <= card_id < card_count else card_count


def option_entity_ids(obs, option, card_count, attack_count):
    yours = int(obs.current.yourIndex)
    player_index = optional_int(option.playerIndex, yours)
    player_index = max(0, min(player_index, len(obs.current.players) - 1))
    candidate = target = None
    if option.type == OptionType.PLAY:
        candidate = area_card(obs, AreaType.HAND, option.index, yours)
    elif option.type in {OptionType.CARD, OptionType.TOOL_CARD,
                          OptionType.ENERGY_CARD, OptionType.ENERGY,
                          OptionType.ABILITY, OptionType.DISCARD}:
        candidate = area_card(obs, option.area, option.index, player_index)
        if option.type == OptionType.TOOL_CARD and candidate is not None:
            cards = list(candidate.tools or [])
            index = optional_int(option.toolIndex, -1)
            candidate = cards[index] if 0 <= index < len(cards) else None
        elif option.type in {OptionType.ENERGY_CARD, OptionType.ENERGY} and candidate is not None:
            cards = list(candidate.energyCards or [])
            index = optional_int(option.energyIndex, -1)
            candidate = cards[index] if 0 <= index < len(cards) else None
    elif option.type in {OptionType.ATTACH, OptionType.EVOLVE}:
        candidate = area_card(obs, option.area, option.index, player_index)
        target = area_card(obs, option.inPlayArea, option.inPlayIndex, yours)
    elif option.type == OptionType.RETREAT:
        active = list(obs.current.players[yours].active or [])
        target = active[0] if active else None
    candidate_id = valid_card_id(candidate, card_count)
    if candidate_id == card_count:
        raw_card_id = optional_int(option.cardId, 0)
        if 0 < raw_card_id < card_count:
            candidate_id = raw_card_id
    target_id = valid_card_id(target, card_count)
    raw_attack_id = option.attackId
    attack_id = (int(raw_attack_id) if raw_attack_id is not None
                 and 0 <= int(raw_attack_id) < attack_count else attack_count)
    return candidate_id, target_id, attack_id


def decoder_features(obs, actions, card_count, attack_count, catalog):
    card_features, attack_damage, _ = catalog
    options = list(obs.select.option)
    option_count = max(1, len(options))
    categorical = torch.empty((len(options), 5), dtype=torch.int64)
    numeric = torch.zeros((len(options), OPTION_NUMERIC_DIM), dtype=torch.float32)
    yours = int(obs.current.yourIndex)
    own_active = active_card(obs.current.players[yours])
    opponent_active = active_card(obs.current.players[1 - yours])
    matchup_known = False
    super_effective = resisted = False
    if own_active is not None and opponent_active is not None:
        own_id = valid_card_id(own_active, card_count)
        opponent_id = valid_card_id(opponent_active, card_count)
        if own_id < card_count and opponent_id < card_count:
            own_energy = card_features[own_id, CARD_ENERGY_TYPE_OFFSET:CARD_ENERGY_TYPE_OFFSET + ENERGY_TYPE_DIM]
            if bool(torch.any(own_energy > 0.5)):
                own_energy_type = int(torch.argmax(own_energy))
                matchup_known = True
                super_effective = bool(card_features[opponent_id, CARD_WEAKNESS_OFFSET + own_energy_type] > 0.5)
                resisted = bool(card_features[opponent_id, CARD_RESISTANCE_OFFSET + own_energy_type] > 0.5)
    context = int(obs.select.context)
    for position, option in enumerate(options):
        candidate_id, target_id, attack_id = option_entity_ids(
            obs, option, card_count, attack_count
        )
        categorical[position] = torch.tensor([
            int(option.type), context, candidate_id, target_id, attack_id,
        ])
        player_index = optional_int(option.playerIndex, yours)
        damage = float(attack_damage[attack_id]) if attack_id < len(attack_damage) else 0.0
        card_type_index = None
        card_type = 0.0
        if candidate_id < card_count:
            card_type_index = int(torch.argmax(card_features[candidate_id, :CARD_TYPE_DIM]))
            card_type = float(card_type_index) / (CARD_TYPE_DIM - 1)
        has_entity = (candidate_id < card_count or target_id < card_count
                      or attack_id < attack_count)
        option_numeric = numeric[position]
        option_numeric[:16] = torch.tensor([
            optional_int(option.number) / 6.0, optional_int(option.index) / 60.0,
            float(player_index == yours), optional_int(option.toolIndex) / 4.0,
            optional_int(option.energyIndex) / 10.0, optional_int(option.count) / 10.0,
            optional_int(option.area) / 12.0, optional_int(option.inPlayArea) / 12.0,
            optional_int(option.inPlayIndex) / 5.0,
            optional_int(option.specialConditionType) / 5.0,
            (position + 1) / option_count, float(has_entity),
            damage, card_type, float(super_effective), float(resisted),
        ])
        player_relation = (0 if option.playerIndex is None else
                           1 if int(option.playerIndex) == yours else 2)
        set_one_hot(option_numeric, 16, 3, player_relation, 'player relation')
        area_index = 0 if option.area is None else int(option.area)
        set_one_hot(option_numeric, 19, 13, area_index, 'area')
        in_play_area_index = (0 if option.inPlayArea is None
                              else int(option.inPlayArea))
        set_one_hot(option_numeric, 32, 13, in_play_area_index, 'in-play area')
        in_play_index = (0 if option.inPlayIndex is None
                         else int(option.inPlayIndex) + 1)
        set_one_hot(option_numeric, 45, 9, in_play_index, 'in-play index')
        special_condition = (0 if option.specialConditionType is None
                             else int(option.specialConditionType) + 1)
        set_one_hot(option_numeric, 54, 6, special_condition, 'special condition')
        set_one_hot(option_numeric, 60, 2, int(has_entity), 'has entity')
        card_type_one_hot = (0 if card_type_index is None
                             else card_type_index + 1)
        set_one_hot(option_numeric, 62, 8, card_type_one_hot, 'card type')
        attack_applicable = attack_id < attack_count and matchup_known
        effectiveness = (0 if not attack_applicable else
                         2 if super_effective else 1)
        resistance = (0 if not attack_applicable else 2 if resisted else 1)
        set_one_hot(option_numeric, 70, 3, effectiveness, 'super effective')
        set_one_hot(option_numeric, 73, 3, resistance, 'resisted')
    action_index = []
    action_offset = [0]
    for action in actions:
        action_index.extend(action)
        action_offset.append(len(action_index))
    return (categorical, numeric,
            torch.tensor(action_index, dtype=torch.int64),
            torch.tensor(action_offset, dtype=torch.int64))


def sparse_tensors(vector):
    return (
        torch.tensor(vector.index, dtype=torch.int64),
        torch.tensor(vector.value, dtype=torch.float32),
        torch.tensor(vector.offset, dtype=torch.int64),
    )


def dense_tensor(values):
    return torch.tensor(values, dtype=torch.float32).unsqueeze(0)


def load_model():
    path = asset_path('model.pt')
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        checkpoint = torch.load(path, map_location='cpu')
    if not isinstance(checkpoint, dict) or 'model' not in checkpoint or 'config' not in checkpoint:
        raise ValueError('Expected checkpoint with model and config keys')
    config = ModelConfig(**checkpoint['config'])
    cards = all_card_data()
    attacks = all_attack()
    catalog = build_numeric_catalog(cards, attacks, config.card_count)
    card_feature_table = catalog[0]
    attack_feature_table = build_attack_feature_table(attacks, config.attack_count)
    model = PTCGTransformer(config, card_feature_table, attack_feature_table)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    return model, config, catalog


MODEL, CONFIG, NUMERIC_CATALOG = load_model()


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return MY_DECK

    actions = enumerate_actions(
        len(obs.select.option), obs.select.minCount, obs.select.maxCount
    )
    if not actions:
        return list(range(min(obs.select.maxCount, len(obs.select.option))))
    encoder, pokemon_appear, own, opponent, global_state = encoder_features(
        obs, MY_DECK, CONFIG.card_count, NUMERIC_CATALOG
    )
    option_categorical, option_numeric, action_index, action_offset = decoder_features(
        obs, actions, CONFIG.card_count, CONFIG.attack_count, NUMERIC_CATALOG
    )
    with torch.inference_mode():
        policy_logits = MODEL(
            *sparse_tensors(encoder),
            torch.tensor(pokemon_appear, dtype=torch.long).unsqueeze(0),
            dense_tensor(own), dense_tensor(opponent),
            dense_tensor(global_state), option_categorical, option_numeric,
            action_index, action_offset,
        )
    return actions[int(policy_logits[0].argmax().item())]


In [ ]:
# Syntax-check the generated agent without importing it yet.
compile(Path('main.py').read_text(), 'main.py', 'exec')
print('main.py syntax OK')

In [ ]:
import tarfile

with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('main.py', arcname='main.py')
    tar.add('deck.csv', arcname='deck.csv')
    tar.add(MODEL_PATH, arcname='model.pt')
    tar.add(CG_PATH, arcname='cg')

size_mb = Path('submission.tar.gz').stat().st_size / 1024**2
print(f'Created submission.tar.gz ({size_mb:.2f} MB)')
with tarfile.open('submission.tar.gz', 'r:gz') as tar:
    print('Archive root:', sorted({name.split('/')[0] for name in tar.getnames()}))